<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [13]:
%%writefile system.txt
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7

Overwriting system.txt


In [14]:
import pathlib

def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    A=[]
    B=[]
    with path.open('r') as f:
        for line in f:
            if not line.strip():
                continue

            tokens=line.split()
            row_A=[]
            semn=1

            for i,token in enumerate(tokens):
                if token=='+':
                    semn=1
                elif token=='-':
                    semn=-1
                elif token=='=':
                    B.append(float(tokens[i+1]))
                    break
                elif 'x' in token or 'z' in token or 'y' in token:
                    coef=token[:-1]
                    if coef=='' or coef=='+':
                        val=1.0
                    elif coef=='-':
                        val=-1.0
                    else:
                        val=float(coef)

                    row_A.append(semn * val)
                    semn=1

            A.append(row_A)

    return A,B

A, B = load_system(pathlib.Path("system.txt"))
print(f"{A=} {B=}")

A=[[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]] B=[5.0, 6.0, 7.0]


### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [15]:
def determinant(matrix: list[list[float]]) -> float:
    return A[0][0]*(A[1][1]*A[2][2]-A[1][2]*A[2][1])-A[0][1]*(A[1][0]*A[2][2]-A[1][2]*A[2][0])+A[0][2]*(A[1][0]*A[2][1]-A[1][1]*A[2][0])

print(f"{determinant(A)=}")

determinant(A)=14.0


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [16]:
def trace(matrix: list[list[float]]) -> float:
    return A[0][0]+A[1][1]+A[2][2]

print(f"{trace(A)=}")

trace(A)=3.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [17]:
import math
def norm(vector: list[float]) -> float:
    return math.sqrt(B[0]**2+B[1]**2+B[2]**2)

print(f"{norm(B)=}")

norm(B)=10.488088481701515


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [18]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    A_T=[]
    rows=len(matrix)
    columns=len(matrix[0])

    for j in range(columns):
       rand_nou=[]
       for i in range(rows):
          rand_nou.append(matrix[i][j])
       A_T.append(rand_nou)

    return A_T

print(f"{transpose(A)=}")

transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [19]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    rez=[]
    rows=len(matrix)
    columns=len(matrix[0])

    for i in range(rows):
      s=0.0
      for j in range(columns):
        s+=matrix[i][j]*vector[j]

      rez.append(s)

    return rez

print(f"{multiply(A, B)=}")

multiply(A, B)=[21.0, 27.0, 35.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [20]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    rez=[]
    det_A=determinant(matrix)
    if det_A==0:
       return []

    for j in range(3):
      A_modified=[row[:] for row in matrix]

      for i in range(3):
          A_modified[i][j]=vector[i]

      det=determinant(A_modified)
      val=det/det_A
      rez.append(val)

    return rez

print(f"{solve_cramer(A, B)=}")

solve_cramer(A, B)=[1.0, 1.0, 1.0]


### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [22]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    rez=[]
    for rows in range(3):
        row=[]
        for column in range(3):
            if rows!=i and column!=j:
                row.append(matrix[rows][column])

        if len(row)>0:
            rez.append(row)

    return rez

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    A_cofactor=[]
    for rows in range(3):
        row=[]
        for column in range(3):
            semn=(-1)**(rows+column)
            minor_matrix=minor(matrix,rows,column)
            det_minor=minor_matrix[0][0]*minor_matrix[1][1]-minor_matrix[0][1]*minor_matrix[1][0]
            element_nou=semn*det_minor
            row.append(element_nou)

        A_cofactor.append(row)

    return A_cofactor


def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    cof_matrix=cofactor(matrix)
    Adjoint=transpose(cof_matrix)
    return Adjoint


def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    det_A=determinant(matrix)
    adj_A=adjoint(matrix)
    if det_A==0:
      return []

    A_inverse=[]
    for i in range(len(adj_A)):
       row=[]
       for j in range(len(adj_A[i])):
          val=adj_A[i][j]/det_A
          row.append(val)

       A_inverse.append(row)

    return multiply(A_inverse,vector)


#print(minor(A,2,2))
print(cofactor(A))
print(f"{solve(A, B)=}")

[[-6.0, 10.0, 4.0], [-7.0, 7.0, 7.0], [11.0, -9.0, -5.0]]
solve(A, B)=[0.35714285714285765, 2.071428571428571, 1.9285714285714288]
